In [1]:
import warnings
import numpy as np
# import lightgbm as lgb
# from fontTools.misc.cython import returns
# from pyarrow.types import is_large_binary
# from sympy.codegen.ast import continue_
# from xgboost import XGBRegressor
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.svm import SVR
# from sklearn.neural_network import MLPRegressor
# from sklearn.tree import DecisionTreeRegressor
# from statsmodels.tools.eval_measures import rmse, hqic_sigma
import pandas as pd
import re
# from optimize_params import *
import matplotlib.pyplot as plt
from datetime import datetime

# np.random.seed(42)

warnings.filterwarnings("ignore", category=RuntimeWarning)

def mape(y_true, y_pred):
    """
    Calculate Mean Absolute Percentage Error (MAPE)

    Parameters:
        y_true (array-like): Actual values
        y_pred (array-like): Predicted values

    Returns:
        float: MAPE in percentage (%)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [2]:
train = pd.read_parquet("data/gold/train.parquet")

train = train.reset_index(drop=True)

In [3]:
import pandas as pd
import numpy as np

def smallest_int_dtype(min_val: int, max_val: int, signed: bool = True) -> str:
    if signed:
        if np.iinfo(np.int8).min <= min_val <= max_val <= np.iinfo(np.int8).max:
            return "int8"
        if np.iinfo(np.int16).min <= min_val <= max_val <= np.iinfo(np.int16).max:
            return "int16"
        if np.iinfo(np.int32).min <= min_val <= max_val <= np.iinfo(np.int32).max:
            return "int32"
        return "int64"
    else:
        if 0 <= min_val <= max_val <= np.iinfo(np.uint8).max:
            return "uint8"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint16).max:
            return "uint16"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint32).max:
            return "uint32"
        return "uint64"


def optimize_df_for_memory(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Converts columns to smaller dtypes.
    For low-decimal float columns, stores scaled integers if that beats float32.
    Returns:
        optimized_df
        metadata dict with scaling info
    """
    df = df.copy()
    meta = {}

    for col in df.columns:
        s = df[col]

        # bool-like columns
        unique_non_null = set(s.dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}):
            if col == "Група":
                df[col] = s.astype("bool")
                meta[col] = {"stored_as": "bool", "scale": 1}
                continue

        # integer columns
        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {"stored_as": dtype, "scale": 1}
            continue

        # float columns
        if pd.api.types.is_float_dtype(s):
            # estimate visible decimal precision
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}
                continue

            decimals = non_null.astype(str).apply(
                lambda x: len(x.split(".")[1].rstrip("0")) if "." in x else 0
            ).max()

            # try scaled integer
            if decimals <= 3:
                scale = 10 ** decimals
                scaled = np.round(s * scale)

                mn = int(np.nanmin(scaled))
                mx = int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))

                int_bytes = np.dtype(int_dtype).itemsize
                float32_bytes = np.dtype("float32").itemsize

                if int_bytes < float32_bytes:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {"stored_as": int_dtype, "scale": scale}
                else:
                    df[col] = s.astype("float32")
                    meta[col] = {"stored_as": "float32", "scale": 1}
            else:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}

    return df, meta

train, train_meta = optimize_df_for_memory(train)

In [13]:
val

,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,GPS-координати - Широта,GPS-координати - Довгота,temperature_2m,...,ОСР опис_Ужгород,ОСР опис_Франківськ,ОСР опис_Харків,ОСР опис_Херсон,ОСР опис_Хмельницький,ОСР опис_ЦЕК,ОСР опис_Черкаси,ОСР опис_Чернівці,ОСР опис_Чернігів,ОСР опис_nan
0,2025,7,1,1,20.000000,1.67615,5.441006,48.279278,26.055273,167,...,False,False,False,False,False,False,False,True,False,False
1,2025,7,1,2,17.000000,1.67615,5.441006,48.279278,26.055273,164,...,False,False,False,False,False,False,False,True,False,False
2,2025,7,1,3,17.000000,1.67615,5.441006,48.279278,26.055273,155,...,False,False,False,False,False,False,False,True,False,False
3,2025,7,1,4,16.000000,1.67615,5.441006,48.279278,26.055273,151,...,False,False,False,False,False,False,False,True,False,False
4,2025,7,1,5,15.000000,1.67615,5.441006,48.279278,26.055273,146,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304494,2025,6,30,20,17.451471,0.81260,4.883898,50.403744,30.684027,172,...,False,False,False,False,False,False,False,False,False,False
304495,2025,6,30,21,17.339781,0.81260,4.883898,50.403744,30.684027,156,...,False,False,False,False,False,False,False,False,False,False
304496,2025,6,30,22,17.311859,0.81260,4.883898,50.403744,30.684027,152,...,False,False,False,False,False,False,False,False,False,False
304497,2025,6,30,23,16.390421,0.81260,4.883898,50.403744,30.684027,149,...,False,False,False,False,False,False,False,False,False,False


In [4]:
y_col = 'Sum of кВт'

In [5]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.6.0.dev20241112+cu121
12.1
True


In [6]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from autogluon.core.metrics import make_scorer
from sklearn.metrics import mean_squared_error

def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))


smape_scorer = make_scorer(
    name="SMAPE",
    score_func=smape,
    optimum=0,
    greater_is_better=False
)

# Optional but strongly recommended:
# keep a smaller subset while debugging
# train = train.sample(1_000_000, random_state=42).reset_index(drop=True)

predictor = TabularPredictor(
    label=y_col,
    problem_type="regression",
    eval_metric=smape_scorer,
    path="models/autogluon_safe"
)

predictor = TabularPredictor(
    label=y_col,
    problem_type="regression",
    eval_metric=smape_scorer,
    path="models/autogluon_gpu",  # new path to avoid any cached state
    verbosity=3,                  # will log "Fitting X with num_gpus: 1"
)

predictor.fit(
    train_data=train,
    presets="best_quality",
    num_gpus=1,
    dynamic_stacking=False,
    num_bag_folds=0,
    num_stack_levels=0,
    # time_limit=5 * 60,
    # hyperparameters=hp,
    ag_args_fit={
        "ag.max_memory_usage_ratio": 2.0,  # allow up to 2x estimated memory
    },
)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.11.14
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
GPU Count:          1
Memory Avail:       15.63 GB / 31.11 GB (50.2%)
Disk Space Avail:   40.36 GB / 475.82 GB (8.5%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'ag.max_memory_usage_ratio': 2.0},
 'auto_stack': True,
 'num_bag_folds': 0,
 'num_bag_sets': 1,
 'num_stack_levels': 0}
Full kwargs:
{'_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'ag.max_memory_usage_ratio': 2.0},
 'auto_stack': True,
 'calibrate': 'auto',
 'ds_args': {'clean_up_fits': True,
             'detection_time_frac': 0.25,
             'enable_ray_logging': True,
             'holdout_data': None,
             'holdout_frac': 

[50]	valid_set's l2: 64.4343	valid_set's SMAPE: -0.318458
[100]	valid_set's l2: 51.2089	valid_set's SMAPE: -0.270543
[150]	valid_set's l2: 45.433	valid_set's SMAPE: -0.239949
[200]	valid_set's l2: 41.593	valid_set's SMAPE: -0.219136
[250]	valid_set's l2: 38.487	valid_set's SMAPE: -0.20838
[300]	valid_set's l2: 35.9435	valid_set's SMAPE: -0.201383
[350]	valid_set's l2: 34.6347	valid_set's SMAPE: -0.196035
[400]	valid_set's l2: 33.3249	valid_set's SMAPE: -0.19132
[450]	valid_set's l2: 32.2149	valid_set's SMAPE: -0.187317
[500]	valid_set's l2: 31.2667	valid_set's SMAPE: -0.1843
[550]	valid_set's l2: 30.5013	valid_set's SMAPE: -0.181029
[600]	valid_set's l2: 29.8092	valid_set's SMAPE: -0.178363
[650]	valid_set's l2: 29.3461	valid_set's SMAPE: -0.17599
[700]	valid_set's l2: 28.8872	valid_set's SMAPE: -0.173444
[750]	valid_set's l2: 28.447	valid_set's SMAPE: -0.171553
[800]	valid_set's l2: 28.0922	valid_set's SMAPE: -0.169669
[850]	valid_set's l2: 27.7014	valid_set's SMAPE: -0.167863
[900]	v

Saving models/autogluon_gpu\models\LightGBMXT\model.pkl
Saving models/autogluon_gpu\utils\attr\LightGBMXT\y_pred_proba_val.pkl
	-0.1029	 = Validation score   (-SMAPE)
	470.57s	 = Training   runtime
	1.95s	 = Validation runtime
	25003.0	 = Inference  throughput (rows/s | 48644 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: LightGBM ... Training model for up to 3081.91s of the 3081.87s of remaining time.
	Fitting LightGBM with 'num_gpus': 1, 'num_cpus': 6
	Training LightGBM with GPU, note that this may negatively impact model quality compared to CPU training.
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'device': 'gpu'}


[50]	valid_set's l2: 61.8561	valid_set's SMAPE: -0.315639
[100]	valid_set's l2: 46.9353	valid_set's SMAPE: -0.265288
[150]	valid_set's l2: 40.4421	valid_set's SMAPE: -0.232654
[200]	valid_set's l2: 36.9828	valid_set's SMAPE: -0.215493
[250]	valid_set's l2: 33.9182	valid_set's SMAPE: -0.20458
[300]	valid_set's l2: 31.9542	valid_set's SMAPE: -0.197474
[350]	valid_set's l2: 30.2066	valid_set's SMAPE: -0.192024
[400]	valid_set's l2: 28.7031	valid_set's SMAPE: -0.187282
[450]	valid_set's l2: 27.2178	valid_set's SMAPE: -0.182933
[500]	valid_set's l2: 26.5964	valid_set's SMAPE: -0.178713
[550]	valid_set's l2: 25.7158	valid_set's SMAPE: -0.175372
[600]	valid_set's l2: 25.076	valid_set's SMAPE: -0.172512
[650]	valid_set's l2: 24.3289	valid_set's SMAPE: -0.169882
[700]	valid_set's l2: 23.8652	valid_set's SMAPE: -0.167335
[750]	valid_set's l2: 23.2205	valid_set's SMAPE: -0.164661
[800]	valid_set's l2: 22.829	valid_set's SMAPE: -0.162445
[850]	valid_set's l2: 22.4543	valid_set's SMAPE: -0.160352
[

Saving models/autogluon_gpu\models\LightGBM\model.pkl
Saving models/autogluon_gpu\utils\attr\LightGBM\y_pred_proba_val.pkl
	-0.0855	 = Validation score   (-SMAPE)
	443.26s	 = Training   runtime
	1.72s	 = Validation runtime
	28218.2	 = Inference  throughput (rows/s | 48644 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: RandomForestMSE ... Training model for up to 2636.57s of the 2636.53s of remaining time.
	Fitting RandomForestMSE with 'num_gpus': 1, 'num_cpus': 12
	Time limit exceeded... Skipping RandomForestMSE.
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: CatBoost ... Training model for up to 2151.87s of the 2151.83s of remaining time.
	Fitting CatBoost with 'num_gpus': 1, 'num_cpus': 6
	Training CatBoost with GPU, note that this may negatively impact model quality compared to CPU training.
	Catboost model hyperparameters: {'iterations': 10000, 'learning_rate': 0.05, 'random_seed': 0, 'allow_writing_files': False, 'eval_metric': 'RMSE', 'th

...

Saving models/autogluon_gpu\models\CatBoost\model.pkl
Saving models/autogluon_gpu\utils\attr\CatBoost\y_pred_proba_val.pkl
	-0.2403	 = Validation score   (-SMAPE)
	31.42s	 = Training   runtime
	0.03s	 = Validation runtime


633:	learn: 6.2109903	test: 6.5832959	best: 6.5832959 (633)	total: 10.3s	remaining: 0us
bestTest = 6.583295905
bestIteration = 633


	1943010.1	 = Inference  throughput (rows/s | 48644 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: ExtraTreesMSE ... Training model for up to 2120.41s of the 2120.37s of remaining time.
	Fitting ExtraTreesMSE with 'num_gpus': 1, 'num_cpus': 12
	Time limit exceeded... Skipping ExtraTreesMSE.
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: NeuralNetFastAI ... Training model for up to 1535.69s of the 1535.66s of remaining time.
	Fitting NeuralNetFastAI with 'num_gpus': 1, 'num_cpus': 6
Fitting Neural Network with parameters {'layers': None, 'emb_drop': 0.1, 'ps': 0.1, 'bs': 'auto', 'lr': 0.01, 'epochs': 'auto', 'early.stopping.min_delta': 0.0001, 'early.stopping.patience': 20, 'smoothing': 0.0}...
Using 0/0 categorical features
Using 540 cont features
		Unable to allocate 18.6 MiB for an array with shape (4864324,) and data type float32
Detailed Traceback:
Traceback (most recent call last):
  File "C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packag

[0]	validation_0-rmse:13.60559	validation_0-custom_metric:0.49153
[50]	validation_0-rmse:7.48126	validation_0-custom_metric:0.28942
[100]	validation_0-rmse:6.91203	validation_0-custom_metric:0.26050
[150]	validation_0-rmse:6.59265	validation_0-custom_metric:0.24002
[200]	validation_0-rmse:6.36395	validation_0-custom_metric:0.22445
[250]	validation_0-rmse:6.19744	validation_0-custom_metric:0.21186
[300]	validation_0-rmse:6.03021	validation_0-custom_metric:0.20097
[350]	validation_0-rmse:5.87207	validation_0-custom_metric:0.19224
[400]	validation_0-rmse:5.75783	validation_0-custom_metric:0.18462
[450]	validation_0-rmse:5.65107	validation_0-custom_metric:0.17818
[500]	validation_0-rmse:5.57912	validation_0-custom_metric:0.17255
[550]	validation_0-rmse:5.51478	validation_0-custom_metric:0.16730
[600]	validation_0-rmse:5.43781	validation_0-custom_metric:0.16296
[650]	validation_0-rmse:5.34280	validation_0-custom_metric:0.15877
[700]	validation_0-rmse:5.27953	validation_0-custom_metric:0.154

C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\xgboost\core.py:160: UserWarning: [17:39:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\xgboost\core.py:160: UserWarning: [17:39:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the 

In [7]:
val = pd.read_parquet("data/gold/val.parquet")
test = pd.read_parquet("data/gold/test.parquet")

val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

val, val_meta = optimize_df_for_memory(val)
test, test_meta = optimize_df_for_memory(test)

In [8]:
lb = predictor.leaderboard(val, silent=True)
print(lb)

val_pred = predictor.predict(val.drop(columns=[y_col]))
test_pred = predictor.predict(test.drop(columns=[y_col]))

print("\nValidation metrics")
print("SMAPE:", smape(val[y_col], val_pred))
print("RMSE :", rmse(val[y_col], val_pred))
print("MAPE :", mape(val[y_col], val_pred))

print("\nTest metrics")
print("SMAPE:", smape(test[y_col], test_pred))
print("RMSE :", rmse(test[y_col], test_pred))
print("MAPE :", mape(test[y_col], test_pred))

Loading: models/autogluon_gpu\models\LightGBMXT\model.pkl
Loading: models/autogluon_gpu\models\LightGBM\model.pkl
Loading: models/autogluon_gpu\models\CatBoost\model.pkl
Loading: models/autogluon_gpu\models\XGBoost\model.pkl
Loading: models/autogluon_gpu\models\WeightedEnsemble_L2\model.pkl


                 model  score_test  score_val eval_metric  pred_time_test  \
0           LightGBMXT   -0.171912  -0.102894       SMAPE       10.170383   
1             LightGBM   -0.178622  -0.085470       SMAPE        9.397026   
2  WeightedEnsemble_L2   -0.179763  -0.065559       SMAPE       27.545195   
3              XGBoost   -0.181329  -0.065779       SMAPE       18.129134   
4             CatBoost   -0.245551  -0.240299       SMAPE        0.200619   

   pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  \
0       1.945526  470.568525                10.170383                1.945526   
1       1.723853  443.262282                 9.397026                1.723853   
2       2.527382  714.008068                 0.019036                0.001001   
3       0.802529  270.715692                18.129134                0.802529   
4       0.025035   31.420970                 0.200619                0.025035   

   fit_time_marginal  stack_level  can_infer  fit_

Loading: models/autogluon_gpu\models\LightGBM\model.pkl
Loading: models/autogluon_gpu\models\XGBoost\model.pkl
Loading: models/autogluon_gpu\models\WeightedEnsemble_L2\model.pkl
Loading: models/autogluon_gpu\models\LightGBM\model.pkl
Loading: models/autogluon_gpu\models\XGBoost\model.pkl
Loading: models/autogluon_gpu\models\WeightedEnsemble_L2\model.pkl



Validation metrics
SMAPE: 0.17976306
RMSE : 9.914105210088263
MAPE : 451793.1

Test metrics
SMAPE: 0.18968086
RMSE : 10.16725140333575
MAPE : 0.32410747


In [9]:
importance = predictor.feature_importance(train)
importance

These features in provided data are not utilized by the predictor and will be ignored: ['EIC-код_62Z1052783389048', 'EIC-код_62Z1426991213062', 'EIC-код_62Z2462903940798', 'EIC-код_62Z2483925041819', 'EIC-код_62Z3459585554968', 'EIC-код_62Z3584090129468', 'EIC-код_62Z4437758189134', 'EIC-код_62Z4948939192262', 'EIC-код_62Z5821574095639', 'EIC-код_62Z5956494916290', 'EIC-код_62Z5987473505748', 'EIC-код_62Z6337618908569', 'EIC-код_62Z6634912359403', 'EIC-код_62Z6814717943705', 'EIC-код_62Z771900708749Y', 'EIC-код_nan', 'АЗС_АЗС_100', 'АЗС_АЗС_69', 'АЗС_АЗС_70', 'АЗС_АЗС_78', 'АЗС_АЗС_83', 'АЗС_АЗС_84', 'АЗС_АЗС_86', 'АЗС_АЗС_87', 'АЗС_АЗС_88', 'АЗС_АЗС_89', 'АЗС_АЗС_90', 'АЗС_АЗС_901', 'АЗС_АЗС_902', 'АЗС_АЗС_91', 'АЗС_АЗС_911', 'АЗС_АЗС_92', 'АЗС_АЗС_93', 'АЗС_АЗС_94', 'АЗС_АЗС_95', 'АЗС_АЗС_99', 'АЗС_nan', 'Тип_ОККО-LPG', 'Тип_nan', 'Область_Чернівецька', 'Область_nan', 'ОСР код_MGA-00200', 'ОСР код_MGA-00400', 'ОСР код_MGA-00500', 'ОСР код_MGA-00600', 'ОСР код_MGA-00700', 'ОСР код_MGA

,importance,stddev,p_value,n,p99_high,p99_low
Hour,0.200087,0.002841,4.872958e-09,5,0.205935,0.194238
GPS-координати - Довгота,0.163726,0.003951,4.066791e-08,5,0.171861,0.155591
apparent_temperature,0.159431,0.009788,1.696316e-06,5,0.179585,0.139277
dew_point_2m,0.112861,0.003281,8.559799e-08,5,0.119617,0.106106
Тип_ОККО-комплекс,0.101933,0.002160,2.419507e-08,5,0.106381,0.097485
...,...,...,...,...,...,...
Область_Одеська,-0.000002,0.000005,7.436924e-01,5,0.000008,-0.000011
АЗС_АЗС_19,-0.000002,0.000018,6.113841e-01,5,0.000035,-0.000040
АЗС_АЗС_68,-0.000019,0.000076,6.959440e-01,5,0.000138,-0.000175
EIC-код_62Z3522594292484,-0.000019,0.000090,6.717813e-01,5,0.000166,-0.000205
